## Transferir o notebook

In [1]:
%pip install -r requirements.txt

import kagglehub
path = kagglehub.dataset_download( "dansbecker/melbourne-housing-snapshot",                                                                   output_dir="data/melbourne",)

print("Ficheiros do dataset em:", path)


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/anytng/Developer/Universidade/Masters/FundamentalsOfAI/FAI26_P02_G09/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ficheiros do dataset em: data/melbourne


## EDA

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

file = "data/melbourne/melb_data.csv"
df = pd.read_csv(file)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)  # datas vêm no formato dd/mm/aaaa

# Só S, SP e SA são vendas concluídas; em PI e VB o Price é uma licitação, não um preço de venda
df["Vendido"] = df["Method"].isin(["S", "SP", "SA"])
vendidos = df[df["Vendido"]]

print(f"{df.shape[0]} linhas x {df.shape[1]} colunas")
print("Vendas entre", df["Date"].min().date(), "e", df["Date"].max().date())
print(f"Vendas concluídas: {len(vendidos)} | não vendidos (PI/VB): {(~df['Vendido']).sum()}")
df.head()

13580 linhas x 22 colunas
Vendas entre 2016-01-28 e 2017-09-23
Vendas concluídas: 10817 | não vendidos (PI/VB): 2763


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Vendido
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0,True
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0,True
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067.0,3.0,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0,True
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04,2.5,3067.0,3.0,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0,False
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04,2.5,3067.0,3.0,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0,False


### Visão geral das colunas
Legenda segundo a descrição do dataset no Kaggle ([Melbourne Housing Snapshot](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot)): snapshot de setembro de 2017 do dataset de Tony Pino, recolhido semanalmente do Domain.com.au. Os imóveis sem preço foram removidos. As unidades vêm da legenda do [dataset original](https://www.kaggle.com/datasets/anthonypino/melbourne-housing-market).

| Coluna | Significado |
|---|---|
| `Rooms` | número de divisões |
| `Price` | preço em dólares australianos |
| `Method` | `S` vendido · `SP` vendido antes do leilão · `SA` vendido após o leilão · `PI` *passed in*, retirado do leilão (não vendido) · `VB` licitação do vendedor (não vendido) |
| `Type` | `h` casa, cottage, villa, semi, terrace · `u` unidade, duplex · `t` townhouse |
| `SellerG` | agência imobiliária |
| `Date` | data da venda |
| `Distance` | distância ao centro (CBD), em km |
| `Postcode`, `Suburb`, `Address` | localização |
| `Bedroom2` | número de quartos recolhido de **outra fonte** (por isso nem sempre coincide com `Rooms`) |
| `Bathroom`, `Car` | casas de banho, lugares de estacionamento |
| `Landsize`, `BuildingArea` | área do terreno e área construída, em m² |
| `YearBuilt` | ano de construção |
| `CouncilArea`, `Regionname` | município e região |
| `Lattitude`, `Longtitude` | coordenadas (os erros ortográficos vêm da origem) |
| `Propertycount` | número de imóveis no subúrbio |

**Atenção:** nas linhas `PI` e `VB` o imóvel não foi vendido, por isso o `Price` não é um preço de venda. As análises de preço abaixo usam só `vendidos`.

In [2]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nao_nulos": df.notna().sum(),
    "distintos": df.nunique(),
    "exemplo": df.iloc[0],
})
overview

,dtype,nao_nulos,distintos,exemplo
Suburb,str,13580,314,Abbotsford
Address,str,13580,13378,85 Turner St
Rooms,int64,13580,9,2
Type,str,13580,3,h
Price,float64,13580,2204,1480000.0
Method,str,13580,5,S
SellerG,str,13580,268,Biggin
Date,datetime64[us],13580,58,2016-12-03 00:00:00
Distance,float64,13580,202,2.5
Postcode,float64,13580,198,3067.0


### Médias e estatísticas descritivas
As estatísticas das características físicas usam todas as linhas. As de preço usam só as vendas concluídas (`vendidos`).

In [3]:
df.describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2)

,count,mean,min,1%,25%,50%,75%,99%,max,std
Rooms,13580.0,2.937997,1.0,1.0,2.0,3.0,3.0,5.0,10.0,0.955748
Price,13580.0,1075684.079455,85000.0,300000.0,650000.0,903000.0,1330000.0,3338150.0,9000000.0,639310.724296
Date,13580,2017-01-28 20:25:03.622975,2016-01-28 00:00:00,2016-04-16 00:00:00,2016-09-10 00:00:00,2017-03-04 00:00:00,2017-06-24 00:00:00,2017-09-23 00:00:00,2017-09-23 00:00:00,NaN
Distance,13580.0,10.137776,0.0,1.6,6.1,9.2,13.0,32.3,48.1,5.868725
Postcode,13580.0,3105.301915,3000.0,3006.0,3044.0,3084.0,3148.0,3429.0,3977.0,90.676964
Bedroom2,13580.0,2.914728,0.0,1.0,2.0,3.0,3.0,5.0,20.0,0.965921
Bathroom,13580.0,1.534242,0.0,1.0,1.0,1.0,2.0,4.0,8.0,0.691712
Car,13518.0,1.610075,0.0,0.0,1.0,2.0,2.0,5.0,10.0,0.962634
Landsize,13580.0,558.416127,0.0,0.0,177.0,440.0,651.0,2959.83,433014.0,3990.669241
BuildingArea,7130.0,151.96765,0.0,7.58,93.0,126.0,174.0,466.42,44515.0,541.014538


In [4]:
# Preço médio e mediano por tipo de imóvel (só vendas concluídas)
vendidos.groupby("Type")["Price"].agg(["count", "mean", "median", "min", "max"]).round(0)

,count,mean,median,min,max
Type,,,,,
h,7652,1202482.0,1042500.0,280000.0,7650000.0
t,873,930354.0,850000.0,319000.0,3475000.0
u,2292,620927.0,582250.0,185000.0,3625000.0


In [5]:
# Preço por região, ordenado pela mediana (só vendas concluídas)
vendidos.groupby("Regionname")["Price"].agg(["count", "mean", "median"]).round(0).sort_values("median")

,count,mean,median
Regionname,,,
Western Victoria,30,382025.0,397500.0
Northern Victoria,39,594821.0,540000.0
Eastern Victoria,46,690413.0,668500.0
Western Metropolitan,2457,869909.0,800000.0
Northern Metropolitan,3212,904036.0,811500.0
South-Eastern Metropolitan,402,901392.0,845000.0
Eastern Metropolitan,1179,1092236.0,1005000.0
Southern Metropolitan,3452,1355475.0,1230000.0


In [6]:
# Médias das características físicas por tipo de imóvel
# Nota: nas unidades (u) o Landsize é muitas vezes o terreno do prédio inteiro (ver "Valores suspeitos")
df.groupby("Type")[["Rooms", "Bathroom", "Car", "Landsize", "BuildingArea", "YearBuilt", "Distance"]].mean().round(1)

,Rooms,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Distance
Type,,,,,,,
h,3.3,1.6,1.8,617.2,176.9,1954.1,11.0
t,2.8,1.8,1.6,279.6,140.0,1999.0,9.9
u,2.0,1.2,1.1,477.3,80.7,1980.0,7.6


In [7]:
# Evolução do preço por trimestre (só vendas concluídas)
vendidos.groupby(vendidos["Date"].dt.to_period("Q"))["Price"].agg(["count", "mean", "median"]).round(0)

,count,mean,median
Date,,,
2016Q1,22,967307.0,919000.0
2016Q2,1452,1018608.0,860500.0
2016Q3,1668,1056299.0,902000.0
2016Q4,1813,1110876.0,935000.0
2017Q1,892,1133515.0,955000.0
2017Q2,2318,1070630.0,922250.0
2017Q3,2652,1005925.0,863000.0


In [8]:
# Price por Method: em PI e VB é o valor de licitação / retirada, não um preço de venda
df.groupby("Method")["Price"].agg(["count", "mean", "median"]).round(0)

,count,mean,median
Method,,,
PI,1564,1133242.0,932500.0
S,9022,1087327.0,925000.0
SA,92,1025772.0,872500.0
SP,1703,899892.0,770000.0
VB,1199,1166510.0,940000.0


In [9]:
# Contagens das variáveis categóricas
for col in ["Type", "Method", "Regionname"]:
    display(df[col].value_counts().to_frame())

,count
Type,
h,9449
u,3017
t,1114


,count
Method,
S,9022
SP,1703
PI,1564
VB,1199
SA,92


,count
Regionname,
Southern Metropolitan,4695
Northern Metropolitan,3890
Western Metropolitan,2948
Eastern Metropolitan,1471
South-Eastern Metropolitan,450
Eastern Victoria,53
Northern Victoria,41
Western Victoria,32


### Valores em falta

In [10]:
missing = pd.DataFrame({
    "em_falta": df.isna().sum(),
    "pct": (df.isna().mean() * 100).round(1),
})
missing[missing["em_falta"] > 0].sort_values("em_falta", ascending=False)

,em_falta,pct
BuildingArea,6450,47.5
YearBuilt,5375,39.6
CouncilArea,1369,10.1
Car,62,0.5


In [11]:
# As faltas sobrepõem-se? (BuildingArea e YearBuilt costumam faltar juntos)
pd.crosstab(df["BuildingArea"].isna(), df["YearBuilt"].isna(),
            rownames=["BuildingArea em falta"], colnames=["YearBuilt em falta"])

YearBuilt em falta,False,True
BuildingArea em falta,,
False,6858,272
True,1347,5103


In [12]:
# As linhas com valores em falta são diferentes das restantes?
for col in ["BuildingArea", "YearBuilt", "CouncilArea"]:
    print(col)
    display(df.groupby(df[col].isna())[["Price", "Distance"]].median()
              .rename(index={False: "preenchido", True: "em falta"}))

BuildingArea


,Price,Distance
BuildingArea,,
preenchido,895000.0,9.2
em falta,917000.0,9.4


YearBuilt


,Price,Distance
YearBuilt,,
preenchido,880000.0,9.2
em falta,937500.0,9.7


CouncilArea


,Price,Distance
CouncilArea,,
preenchido,895000.0,9.2
em falta,1010000.0,12.3


In [13]:
# Quase todas as CouncilArea em falta podem ser preenchidas a partir do Suburb
council_por_suburb = df.dropna(subset=["CouncilArea"]).groupby("Suburb")["CouncilArea"].agg(lambda s: s.mode()[0])
sem_council = df["CouncilArea"].isna()
print(f"CouncilArea em falta: {sem_council.sum()}")
print(f"Recuperáveis pelo Suburb: {df.loc[sem_council, 'Suburb'].isin(council_por_suburb.index).sum()}")

CouncilArea em falta: 1369
Recuperáveis pelo Suburb: 1362


### Duplicados

In [14]:
print("Linhas exatamente iguais:", df.duplicated().sum())

# Mesmo imóvel, mesma data -> registo duplicado (ex.: a mesma agência com dois nomes)
same_day = df[df.duplicated(["Address", "Suburb", "Date"], keep=False)]
print("Linhas com mesma morada + data:", len(same_day))
same_day[["Address", "Suburb", "Date", "Price", "Method", "SellerG"]]

Linhas exatamente iguais: 0
Linhas com mesma morada + data: 2


,Address,Suburb,Date,Price,Method,SellerG
7769,69 Victoria St,Coburg,2017-04-29,720000.0,PI,C21
7770,69 Victoria St,Coburg,2017-04-29,720000.0,PI,Century


### Imóveis vendidos mais do que uma vez
Um imóvel é identificado por `Address` + `Suburb`. Nem todas as linhas são vendas concluídas:
`PI` e `VB` significam que o imóvel **não foi vendido**, por isso separamos "aparece mais do que uma vez" de "foi mesmo vendido mais do que uma vez".

In [15]:
key = ["Address", "Suburb"]
repetidos = df[df.duplicated(key, keep=False)].sort_values(key + ["Date"])

print("Imóveis que aparecem mais do que uma vez:", repetidos.groupby(key).ngroups,
      f"({len(repetidos)} linhas)")
repetidos[key + ["Date", "Price", "Method", "SellerG", "Type", "Rooms", "Landsize", "BuildingArea"]]

Imóveis que aparecem mais do que uma vez: 112 (226 linhas)


,Address,Suburb,Date,Price,Method,SellerG,Type,Rooms,Landsize,BuildingArea
4958,1 Bellarine St,Preston,2017-03-18,1400000.0,VB,Jellis,h,4,559.0,217.0
9281,1 Bellarine St,Preston,2017-06-03,1400000.0,PI,hockingstuart,h,5,558.0,217.0
7802,1 Daisy St,Essendon,2017-04-08,1400000.0,VB,Nelson,h,4,267.0,NaN
9144,1 Daisy St,Essendon,2017-06-03,1360000.0,S,Jellis,h,4,267.0,NaN
8055,1/1 Clarendon St,Maidstone,2017-04-22,675000.0,S,Biggin,t,3,159.0,128.0
...,...,...,...,...,...,...,...,...,...,...
5186,9/99 Barton St,Reservoir,2016-08-22,335000.0,S,Love,u,2,0.0,54.0
3227,97 Edwin St,Heidelberg Heights,2016-09-03,815000.0,SP,Fletchers,h,2,589.0,120.0
11069,97 Edwin St,Heidelberg Heights,2017-08-12,1645000.0,PI,Fletchers,h,4,596.0,120.0
6842,99 Neill St,Carlton,2016-08-22,1550000.0,VB,Nelson,h,3,0.0,192.0


In [16]:
# Apenas vendas concluídas (S, SP, SA)
revendidos = vendidos[vendidos.duplicated(key, keep=False)].sort_values(key + ["Date"])
print("Imóveis efetivamente vendidos mais do que uma vez:", revendidos.groupby(key).ngroups)
revendidos[key + ["Date", "Price", "Method", "SellerG"]]

Imóveis efetivamente vendidos mais do que uma vez: 35


,Address,Suburb,Date,Price,Method,SellerG
8055,1/1 Clarendon St,Maidstone,2017-04-22,675000.0,S,Biggin
12069,1/1 Clarendon St,Maidstone,2017-07-29,670000.0,SP,Jas
7229,1/12 Hatfield Ct,West Footscray,2016-05-07,520000.0,SP,Sweeney
7237,1/12 Hatfield Ct,West Footscray,2016-09-10,530000.0,SP,Sweeney
12510,1/16 Osborne Av,Glen Iris,2017-09-09,965000.0,S,hockingstuart
...,...,...,...,...,...,...
8648,81 Carter St,Middle Park,2017-04-08,1335000.0,S,Greg
8801,9/44 Fletcher St,Essendon,2017-07-01,310000.0,S,Considine
10663,9/44 Fletcher St,Essendon,2017-07-08,310000.0,SP,Considine
5119,9/99 Barton St,Reservoir,2016-08-13,335000.0,SP,Love


In [17]:
# Variação de preço e tempo entre a primeira e a última venda concluída de cada imóvel revendido
evolucao = revendidos.groupby(key).agg(
    vendas=("Price", "size"),
    primeira_data=("Date", "first"),
    ultima_data=("Date", "last"),
    primeiro_preco=("Price", "first"),
    ultimo_preco=("Price", "last"),
)
evolucao["dias"] = (evolucao["ultima_data"] - evolucao["primeira_data"]).dt.days
evolucao["variacao_pct"] = ((evolucao["ultimo_preco"] / evolucao["primeiro_preco"] - 1) * 100).round(1)
display(evolucao[["dias", "variacao_pct"]].describe().round(1))
evolucao.sort_values("variacao_pct", ascending=False).head(10)

,dias,variacao_pct
count,35.0,35.0
mean,104.3,7.4
std,140.0,18.2
min,5.0,-1.0
25%,7.0,0.0
50%,9.0,0.0
75%,174.0,4.8
max,441.0,93.3


,,vendas,primeira_data,ultima_data,primeiro_preco,ultimo_preco,dias,variacao_pct
Address,Suburb,,,,,,,
18 Molesworth Ct,West Footscray,2,2016-05-28,2017-08-12,660000.0,1276000.0,441,93.3
45 Stanley St,Glenroy,2,2016-08-06,2017-08-12,852500.0,1220000.0,371,43.1
101 The Fairway,Kingsbury,2,2016-11-12,2017-06-03,610000.0,800001.0,203,31.1
81 Carter St,Middle Park,2,2016-05-14,2017-04-08,1020000.0,1335000.0,329,30.9
28 Guildford Rd,Surrey Hills,2,2016-09-03,2017-09-23,2009000.0,2350000.0,385,17.0
10 Beryl St,Essendon West,2,2016-05-07,2017-06-24,1245000.0,1390000.0,413,11.6
15 Ellendale St,Balwyn North,2,2016-05-28,2016-08-06,1038000.0,1150000.0,70,10.8
3 Brian St,Fawkner,2,2016-12-10,2017-04-08,625000.0,660000.0,119,5.6
3/9 Poplar Gr,Carnegie,2,2016-06-04,2017-04-08,290000.0,305000.0,308,5.2


In [18]:
# Imóveis repetidos cujas características físicas mudam entre registos (sinal de erro de introdução)
# Bedroom2 fica de fora porque vem de outra fonte e pode variar sem ser erro
fisicas = ["Rooms", "Type", "Bathroom", "Car", "Landsize", "BuildingArea", "YearBuilt"]
inconsistentes = repetidos.groupby(key)[fisicas].nunique().gt(1).any(axis=1)
print("Imóveis repetidos com características diferentes entre registos:", inconsistentes.sum())
repetidos.set_index(key).loc[inconsistentes[inconsistentes].index, ["Date"] + fisicas]

Imóveis repetidos com características diferentes entre registos: 43


Date  Rooms Type  Bathroom  Car  Landsize  BuildingArea  YearBuilt
Address            Suburb                                                                                     
1 Bellarine St     Preston            2017-03-18      4    h       2.0  2.0     559.0         217.0     1925.0
                   Preston            2017-06-03      5    h       2.0  2.0     558.0         217.0     1925.0
1/108 Truganini Rd Carnegie           2016-05-14      3    t       2.0  2.0     325.0         223.0     2005.0
                   Carnegie           2017-07-08      4    t       3.0  2.0     326.0         223.0     2005.0
1/12 Hatfield Ct   West Footscray     2016-05-07      6    u       2.0  1.0       0.0         171.0     1970.0
...                                          ...    ...  ...       ...  ...       ...           ...        ...
75 Maribyrnong Rd  Ascot Vale         2017-05-13      6    h       3.0  2.0    1012.0         435.0     1887.0
9/11 Smith St      Kensington         2017-05-13      2    u       1.0  1.0       0.0          73.0     2011.0
                   Kensington         2017-07-01      2    u       1.0  1.0       0.0          70.0     2011.0
97 Edwin St        Heidelberg Heights 2016-09-03      2    h       1.0  2.0     589.0         120.0     1930.0
                   Heidelberg Heights 2017-08-12      4    h       2.0  4.0     596.0         120.0     1930.0

[87 rows x 8 columns]

### Valores suspeitos
Um valor só é suspeito em relação ao tipo de imóvel. `Landsize = 0` é normal numa unidade (não tem terreno próprio), mas não numa casa. Por isso cada verificação é vista por `Type` e, quando faz sentido, por região (as regiões "Victoria" são zonas rurais, a 30-50 km do centro).

Nesta secção só se identificam os casos; nenhum valor é alterado.

In [19]:
# Distribuições por tipo de imóvel: o que é "normal" para cada tipo
for col in ["Landsize", "BuildingArea", "Rooms", "Bathroom", "YearBuilt"]:
    print(col)
    display(df.groupby("Type")[col].describe(percentiles=[.01, .05, .5, .95, .99]).round(0))

Landsize


,count,mean,std,min,1%,5%,50%,95%,99%,max
Type,,,,,,,,,,
h,9449.0,617.0,4676.0,0.0,0.0,132.0,560.0,903.0,1411.0,433014.0
t,1114.0,280.0,656.0,0.0,0.0,0.0,198.0,696.0,2032.0,15900.0
u,3017.0,477.0,1729.0,0.0,0.0,0.0,0.0,2166.0,6811.0,37000.0


BuildingArea


,count,mean,std,min,1%,5%,50%,95%,99%,max
Type,,,,,,,,,,
h,4860.0,177.0,651.0,0.0,22.0,80.0,144.0,322.0,520.0,44515.0
t,686.0,140.0,148.0,0.0,2.0,28.0,130.0,233.0,296.0,3558.0
u,1584.0,81.0,51.0,1.0,25.0,41.0,75.0,136.0,191.0,1561.0


Rooms


,count,mean,std,min,1%,5%,50%,95%,99%,max
Type,,,,,,,,,,
h,9449.0,3.0,1.0,1.0,2.0,2.0,3.0,5.0,5.0,10.0
t,1114.0,3.0,1.0,1.0,2.0,2.0,3.0,4.0,4.0,5.0
u,3017.0,2.0,1.0,1.0,1.0,1.0,2.0,3.0,3.0,8.0


Bathroom


,count,mean,std,min,1%,5%,50%,95%,99%,max
Type,,,,,,,,,,
h,9449.0,2.0,1.0,0.0,1.0,1.0,1.0,3.0,4.0,8.0
t,1114.0,2.0,1.0,0.0,1.0,1.0,2.0,3.0,3.0,4.0
u,3017.0,1.0,0.0,0.0,1.0,1.0,1.0,2.0,2.0,4.0


YearBuilt


,count,mean,std,min,1%,5%,50%,95%,99%,max
Type,,,,,,,,,,
h,5408.0,1954.0,38.0,1196.0,1880.0,1890.0,1960.0,2010.0,2015.0,2018.0
t,762.0,1999.0,20.0,1860.0,1909.0,1960.0,2005.0,2015.0,2016.0,2017.0
u,2035.0,1980.0,25.0,1830.0,1900.0,1940.0,1975.0,2012.0,2014.0,2017.0


In [20]:
rural = df["Regionname"].str.contains("Victoria")
checks = {
    "Landsize == 0": df["Landsize"] == 0,
    "Landsize > 2000 m2": df["Landsize"] > 2000,
    "Landsize > 10000 m2": df["Landsize"] > 10000,
    "BuildingArea <= 1 m2": df["BuildingArea"] <= 1,
    "BuildingArea < 30 m2": df["BuildingArea"] < 30,
    "BuildingArea > 600 m2": df["BuildingArea"] > 600,
    "BuildingArea ~= Landsize (±2%)": (df["Landsize"] > 0) & ((df["BuildingArea"] / df["Landsize"] - 1).abs() <= 0.02),
    "Bathroom == 0": df["Bathroom"] == 0,
    "Bedroom2 == 0": df["Bedroom2"] == 0,
    "Rooms >= 7": df["Rooms"] >= 7,
    "YearBuilt < 1850": df["YearBuilt"] < 1850,
    "YearBuilt > ano da venda": df["YearBuilt"] > df["Date"].dt.year,
    "Price < 200k": df["Price"] < 200_000,
}
tabela = pd.DataFrame({k: v.groupby(df["Type"]).sum() for k, v in checks.items()}).T
tabela["total"] = tabela.sum(axis=1)
tabela["rural"] = [(v & rural).sum() for v in checks.values()]
tabela

Type,h,t,u,total,rural
Landsize == 0,171,139,1629,1939,1
Landsize > 2000 m2,50,12,176,238,9
Landsize > 10000 m2,10,1,15,26,5
BuildingArea <= 1 m2,18,6,4,28,0
BuildingArea < 30 m2,55,36,19,110,1
BuildingArea > 600 m2,31,2,1,34,1
BuildingArea ~= Landsize (±2%),81,21,16,118,2
Bathroom == 0,15,2,17,34,0
Bedroom2 == 0,11,2,3,16,0
Rooms >= 7,18,0,1,19,0


#### Landsize = 0
A maioria são unidades e townhouses, que partilham o terreno do prédio ou condomínio, por isso o 0 faz sentido. Numa casa (`h`), um terreno de 0 m² não faz sentido: parte dessas moradas tem `/` (ex.: `1/5 Little St`), o que sugere que são na verdade unidades mal classificadas; mesmo assim, o 0 pode ser real (ex.: imóveis em terreno partilhado), por isso fica como está.

In [21]:
display(df.loc[checks["Landsize == 0"], "Type"].value_counts().to_frame())

casas_sem_terreno = df[checks["Landsize == 0"] & (df["Type"] == "h")]
print(f"Casas com Landsize = 0: {len(casas_sem_terreno)}, "
      f"das quais com '/' na morada: {casas_sem_terreno['Address'].str.contains('/').sum()}")
casas_sem_terreno[["Suburb", "Address", "Rooms", "Price", "BuildingArea", "Distance"]].head(15)

,count
Type,
u,1629
h,171
t,139


Casas com Landsize = 0: 171, das quais com '/' na morada: 55


,Suburb,Address,Rooms,Price,BuildingArea,Distance
30,Abbotsford,31 Turner St,4,1542000.0,NaN,2.5
39,Airport West,50 Bedford St,3,730000.0,145.0,13.5
40,Airport West,50 Bedford St,3,770000.0,145.0,13.5
52,Airport West,13 Etzel St,3,805000.0,105.0,13.5
95,Albert Park,4a Gatehouse La,3,1370000.0,85.0,3.3
111,Albert Park,2b Dundas Pl,3,1532000.0,NaN,3.3
118,Albert Park,74 Bridport St,3,2800000.0,NaN,3.3
193,Altona North,1/5 Little St,3,720000.0,130.0,11.1
243,Armadale,4/20 Mercer Rd,2,1780000.0,NaN,6.3
302,Ascot Vale,2/78 Ormond Rd,2,636500.0,NaN,5.9


#### Landsize grande em unidades
Nas unidades com `Landsize` elevado, o valor é o terreno do prédio inteiro e não o da unidade: unidades diferentes do mesmo prédio repetem exatamente o mesmo `Landsize`. Não é um erro, mas significa que o `Landsize` de uma unidade não é comparável com o de uma casa.

In [22]:
u = df[(df["Type"] == "u") & df["Address"].str.contains("/") & (df["Landsize"] > 0)].copy()
u["predio"] = u["Address"].str.split("/").str[1] + ", " + u["Suburb"]
predios = u.groupby("predio").agg(unidades=("Landsize", "size"), landsize_distintos=("Landsize", "nunique"),
                                  landsize=("Landsize", "median"))
predios = predios[predios["unidades"] >= 2]
print(f"Prédios com 2+ unidades vendidas: {len(predios)}, "
      f"com o mesmo Landsize em todas: {(predios['landsize_distintos'] == 1).sum()}")
predios[predios["landsize_distintos"] == 1].sort_values("landsize", ascending=False).head(15)

Prédios com 2+ unidades vendidas: 80, com o mesmo Landsize em todas: 30


,unidades,landsize_distintos,landsize
predio,,,
"800 Chapel St, South Yarra",2,1,17200.0
"79 River St, South Yarra",2,1,17200.0
"61 Stawell St, Richmond",2,1,7455.0
"41 Murrumbeena Rd, Murrumbeena",2,1,5289.0
"127 Hawdon St, Heidelberg",2,1,4497.0
"72 Altona St, Kensington",2,1,3957.0
"152 Sturt St, Southbank",3,1,3878.0
"3 Hoddle St, Collingwood",2,1,3701.0
"71 Henry St, Kensington",3,1,2592.0


#### Landsize > 10 000 m² (1 hectare)
Em zonas rurais (Gisborne, Silvan, a 35-48 km do centro) terrenos de vários hectares são plausíveis. Nas unidades, é o terreno do prédio (ver acima). Em casas dentro da área metropolitana é muito suspeito: o caso extremo é uma casa de 3 divisões em Fitzroy, a 2 km do centro, com 433 014 m² (43 hectares).

In [23]:
df.loc[checks["Landsize > 10000 m2"],
       ["Suburb", "Regionname", "Type", "Rooms", "Price", "Landsize", "BuildingArea", "Distance"]
      ].assign(rural=rural).sort_values(["Type", "Landsize"])

,Suburb,Regionname,Type,Rooms,Price,Landsize,BuildingArea,Distance,rural
2487,Essendon North,Western Metropolitan,h,3,1425000.0,10100.0,NaN,9.4,False
12340,Kew East,Southern Metropolitan,h,4,1600000.0,12300.0,NaN,7.3,False
12504,Gisborne,Northern Victoria,h,4,780000.0,20200.0,189.0,45.9,True
10045,Silvan,Eastern Victoria,h,4,1070000.0,38490.0,NaN,34.6,True
11371,Gisborne,Northern Victoria,h,4,807000.0,40468.0,NaN,45.9,True
5194,Reservoir,Northern Metropolitan,h,3,572000.0,41400.0,NaN,11.2,False
13245,New Gisborne,Northern Victoria,h,5,1355000.0,44500.0,44515.0,48.1,True
687,Balwyn North,Southern Metropolitan,h,3,2000000.0,75100.0,NaN,9.2,False
10504,Silvan,Eastern Victoria,h,3,1085000.0,76000.0,NaN,34.6,True
11020,Fitzroy,Northern Metropolitan,h,3,2700000.0,433014.0,NaN,2.1,False


#### BuildingArea igual ao Landsize
Em vários imóveis a área construída é praticamente igual à área do terreno. Numa townhouse de vários pisos num terreno pequeno isso é possível, mas numa casa com centenas de m² de terreno é mais provável que o `Landsize` tenha sido copiado para `BuildingArea`.

In [24]:
copia = df[checks["BuildingArea ~= Landsize (±2%)"]]
display(copia["Type"].value_counts().to_frame())
copia.loc[copia["Landsize"] >= 400, ["Suburb", "Type", "Rooms", "Bathroom", "Price", "Landsize", "BuildingArea"]
         ].sort_values("Landsize")

,count
Type,
h,81
t,21
u,16


,Suburb,Type,Rooms,Bathroom,Price,Landsize,BuildingArea
9743,Taylors Hill,h,3,2.0,558000.0,431.0,431.0000
8988,Williams Landing,h,4,2.0,510000.0,443.0,443.0000
5043,Reservoir,h,3,2.0,1005000.0,466.0,467.0000
4691,Pascoe Vale,h,3,2.0,960000.0,501.0,501.0000
12456,Brunswick West,h,2,1.0,1225000.0,501.0,495.0000
4984,Preston,h,3,1.0,1123000.0,504.0,506.0000
12875,West Footscray,h,4,3.0,1060000.0,512.0,512.0000
5133,Reservoir,h,4,2.0,811000.0,525.0,525.0000
4990,Preston,h,2,1.0,900000.0,529.0,528.0000
7920,Hadfield,h,4,3.0,880000.0,530.0,530.0000


#### BuildingArea muito pequena
Valores de 0 ou 1 m² em casas com 3-5 divisões são claramente valores em falta. Abaixo de ~30 m² depende do tipo: num estúdio (`u` com 1 divisão) é possível, numa casa não. A área por divisão ajuda a comparar imóveis de tamanhos diferentes.

In [25]:
m2_por_divisao = (df["BuildingArea"] / df["Rooms"]).where(df["BuildingArea"] > 0)
display(m2_por_divisao.groupby(df["Type"]).describe(percentiles=[.01, .05, .5, .99]).round(1))

df.loc[checks["BuildingArea < 30 m2"], ["Suburb", "Type", "Rooms", "Bathroom", "Price", "Landsize", "BuildingArea"]
      ].assign(m2_por_divisao=m2_por_divisao.round(1)).sort_values(["Type", "BuildingArea"])

,count,mean,std,min,1%,5%,50%,99%,max
Type,,,,,,,,,
h,4846.0,52.1,131.6,0.2,8.8,30.5,45.5,149.3,8903.0
t,683.0,50.8,71.2,0.3,1.0,24.5,46.7,106.6,1779.0
u,1584.0,42.2,23.4,0.5,8.7,28.0,40.0,85.2,780.5


,Suburb,Type,Rooms,Bathroom,Price,Landsize,BuildingArea,m2_por_divisao
12226,Balwyn North,h,5,3.0,1950000.0,743.0,0.0,NaN
12249,Bundoora,h,3,1.0,814000.0,542.0,0.0,NaN
12395,Roxburgh Park,h,4,2.0,570000.0,504.0,0.0,NaN
13040,Prahran,h,3,2.0,1390000.0,125.0,0.0,NaN
13207,Huntingdale,h,3,2.0,1205000.0,622.0,0.0,NaN
...,...,...,...,...,...,...,...,...
8364,South Yarra,u,3,2.0,1320000.0,15.0,15.0,5.0
11197,South Yarra,u,3,2.0,940000.0,0.0,20.0,6.7
8811,Footscray,u,1,1.0,170000.0,30.0,26.0,26.0
22,Abbotsford,u,3,2.0,1090000.0,4290.0,27.0,9.0


#### BuildingArea muito grande
Casas grandes com 5+ divisões e 600-1000 m² são plausíveis em subúrbios caros. Uma casa de dois pisos pode ter mais área construída do que terreno, mas não várias vezes mais: 6 791 m² num terreno de 732 m², ou acima de 3 000 m², não são plausíveis.

In [26]:
df.loc[checks["BuildingArea > 600 m2"], ["Suburb", "Type", "Rooms", "Bathroom", "Price", "Landsize", "BuildingArea", "YearBuilt"]
      ].sort_values("BuildingArea")

,Suburb,Type,Rooms,Bathroom,Price,Landsize,BuildingArea,YearBuilt
204,Altona North,h,3,1.0,770000.0,602.0,603.0000,NaN
2848,Glen Iris,h,3,1.0,1420000.0,615.0,607.0000,1940.0
3073,Hampton,h,4,2.0,1515000.0,611.0,613.0000,1950.0
379,Ashburton,h,8,7.0,2950000.0,1472.0,618.0000,2009.0
10968,Clarinda,h,4,2.0,1241000.0,625.0,625.0000,1950.0
10172,Bundoora,h,3,2.0,1405000.0,639.0,634.0000,NaN
5990,Sunshine,h,3,2.0,910000.0,653.0,650.0000,1992.0
13180,Forest Hill,h,4,2.0,1350000.0,653.0,653.0000,1970.0
11425,Meadow Heights,h,4,2.0,570000.0,665.0,665.0000,NaN
2864,Glen Iris,h,5,3.0,2400000.0,709.0,672.0000,1930.0


#### Bathroom = 0 e Bedroom2 = 0
Qualquer habitação tem pelo menos uma casa de banho, por isso `Bathroom = 0` não faz sentido em nenhum tipo. Estas linhas também não têm `BuildingArea` nem `YearBuilt`, o que indica uma recolha incompleta. São poucas linhas e ficam como estão. `Bedroom2 = 0` pode fazer sentido num estúdio (`u`), mas não numa casa; e como `Bedroom2` vem de outra fonte, `Rooms` é a referência mais fiável.

In [27]:
df.loc[checks["Bathroom == 0"] | checks["Bedroom2 == 0"],
       ["Suburb", "Type", "Rooms", "Bedroom2", "Bathroom", "Price", "BuildingArea", "YearBuilt", "Method"]
      ].sort_values(["Type", "Rooms"])

,Suburb,Type,Rooms,Bedroom2,Bathroom,Price,BuildingArea,YearBuilt,Method
584,Balwyn,h,2,2.0,0.0,1010000.0,NaN,NaN,PI
2777,Glen Iris,h,2,2.0,0.0,845000.0,NaN,NaN,S
3787,Malvern,h,2,2.0,0.0,780000.0,NaN,NaN,VB
6671,Yarraville,h,2,2.0,0.0,610000.0,NaN,NaN,SP
9917,Hawthorn,h,2,0.0,0.0,1480000.0,NaN,NaN,S
139,Alphington,h,3,3.0,0.0,1485000.0,NaN,NaN,SP
827,Bentleigh East,h,3,0.0,1.0,1355000.0,NaN,NaN,S
913,Bentleigh East,h,3,0.0,0.0,700000.0,NaN,NaN,S
1063,Brighton,h,3,3.0,0.0,1900000.0,NaN,NaN,S
6170,Surrey Hills,h,3,0.0,2.0,1560000.0,209.0,2013.0,S


#### YearBuilt
- **1196** é quase certamente um erro de escrita (provavelmente 1996).
- **1830** numa unidade em Collingwood é plausível: um edifício antigo convertido em apartamentos.
- **YearBuilt depois da data de venda** também é plausível: são vendas em planta (*off-the-plan*), em que o imóvel é vendido antes de estar construído.

In [28]:
df.loc[checks["YearBuilt < 1850"] | checks["YearBuilt > ano da venda"],
       ["Suburb", "Address", "Type", "YearBuilt", "Date", "Method", "Price"]].sort_values("YearBuilt")

,Suburb,Address,Type,YearBuilt,Date,Method,Price
9968,Mount Waverley,5 Armstrong St,h,1196.0,2017-06-24,VB,1200000.0
2079,Collingwood,2/79 Oxford St,u,1830.0,2016-09-03,S,855000.0
441,Avondale Heights,157 Canning St,t,2017.0,2016-12-03,S,851000.0
3489,Keilor East,20 Keith Gr,h,2017.0,2016-04-23,S,825000.0
4282,Newport,7 Durkin St,h,2017.0,2016-05-28,S,945000.0
5153,Reservoir,89 Darebin Bvd,h,2017.0,2016-09-17,S,767500.0
7060,Heidelberg,36 Buckingham Dr,h,2017.0,2016-07-30,S,1030000.0
1234,Brighton East,8 Thomas St,h,2018.0,2016-09-24,S,1310000.0


#### Preço muito baixo
A maioria dos preços abaixo de 200 mil são linhas `PI` ou `VB`, ou seja, licitações e não vendas. As vendas reais que restam são unidades de 1-2 divisões em subúrbios mais baratos do oeste (ex.: Albion), o que é plausível.

In [29]:
df.loc[checks["Price < 200k"], ["Suburb", "Regionname", "Type", "Rooms", "Price", "Method", "Vendido", "Landsize"]
      ].sort_values("Price")

,Suburb,Regionname,Type,Rooms,Price,Method,Vendido,Landsize
2652,Footscray,Western Metropolitan,u,1,85000.0,PI,False,0.0
1805,Caulfield,Southern Metropolitan,h,4,131000.0,PI,False,499.0
1927,Coburg,Northern Metropolitan,h,4,145000.0,PI,False,536.0
7303,Albion,Western Metropolitan,u,1,145000.0,PI,False,36.0
7940,Hawthorn,Southern Metropolitan,u,1,160000.0,VB,False,322.0
8811,Footscray,Western Metropolitan,u,1,170000.0,PI,False,30.0
12666,Brunswick,Northern Metropolitan,u,1,170000.0,VB,False,1250.0
7293,Albion,Western Metropolitan,u,1,185000.0,S,True,2347.0
8504,West Footscray,Western Metropolitan,u,1,185000.0,PI,False,0.0
7305,Albion,Western Metropolitan,u,2,190000.0,SP,True,0.0


#### Muitas divisões
Casas com 7-10 divisões são plausíveis quando acompanhadas de várias casas de banho e terrenos grandes. São mais estranhas quando a área construída é pequena (ex.: 8 divisões em 101 m²), o que pode indicar casas partilhadas (*rooming houses*) ou um erro. A unidade com 8 divisões a 2,25 milhões pode ser a venda de um bloco inteiro.

In [30]:
df.loc[checks["Rooms >= 7"], ["Suburb", "Type", "Rooms", "Bedroom2", "Bathroom", "Price", "Landsize", "BuildingArea"]
      ].assign(m2_por_divisao=m2_por_divisao.round(1)).sort_values("Rooms")

,Suburb,Type,Rooms,Bedroom2,Bathroom,Price,Landsize,BuildingArea,m2_por_divisao
6881,Essendon West,h,7,5.0,2.0,1300000.0,614.0,316.00,45.1
589,Balwyn North,h,7,7.0,2.0,1350000.0,942.0,NaN,NaN
636,Balwyn North,h,7,5.0,3.0,2600000.0,637.0,529.00,75.6
707,Bentleigh,h,7,7.0,3.0,1005000.0,607.0,NaN,NaN
1417,Brunswick West,h,7,7.0,3.0,1205000.0,447.0,287.00,41.0
11207,Strathmore,h,7,7.0,3.0,2000000.0,608.0,355.00,50.7
11092,Kew,h,7,7.0,4.0,4050000.0,725.0,NaN,NaN
10177,Burwood,h,7,7.0,4.0,1642000.0,765.0,146.00,20.9
12129,Preston,h,7,7.0,3.0,1350000.0,530.0,NaN,NaN
8793,Donvale,h,7,7.0,2.0,2705000.0,5022.0,409.54,58.5


### Resumo dos valores suspeitos
A maioria dos casos é plausível para o tipo de imóvel, ou são poucas linhas, por isso **fica como está**: remover demasiados dados prejudicaria a classificação e o clustering.

Limpeza mínima proposta, só para o que é claramente impossível (ainda não aplicada):

| Caso | Proposta |
|---|---|
| `YearBuilt = 1196` | Valor em falta |
| `BuildingArea` de 0 ou 1 m² | Valor em falta |
| Registo duplicado (69 Victoria St, Coburg, mesma data) | Remover uma das duas linhas |
| `Price` em linhas `PI`/`VB` | Manter as linhas, mas não usar como preço de venda nas análises de preço |

Tudo o resto (`Landsize = 0`, terrenos grandes, `Bathroom = 0`, imóveis com muitas divisões, etc.) fica como está.

## Preparação dos dados

### Limpeza
Limpeza mínima, só do que é claramente impossível (ver o resumo dos valores suspeitos). Todo o resto, incluindo `Landsize = 0` e as linhas `PI`/`VB`, fica como está.

In [31]:
dados = df.copy()
n_inicial = len(dados)

# 1. Registo duplicado: mesmo imóvel na mesma data
dados = dados.drop_duplicates(subset=["Address", "Suburb", "Date"], keep="first")

# 2. YearBuilt impossível (1196) -> em falta
ano_invalido = dados["YearBuilt"] < 1800
dados.loc[ano_invalido, "YearBuilt"] = np.nan

# 3. BuildingArea de 0 ou 1 m² -> em falta
area_invalida = dados["BuildingArea"] <= 1
dados.loc[area_invalida, "BuildingArea"] = np.nan

print(f"Linhas removidas: {n_inicial - len(dados)} ({(n_inicial - len(dados)) / n_inicial:.3%})")
print(f"YearBuilt passados a em falta: {ano_invalido.sum()}")
print(f"BuildingArea passados a em falta: {area_invalida.sum()}")
print(f"Linhas finais: {len(dados)}")

Linhas removidas: 1 (0.007%)
YearBuilt passados a em falta: 1
BuildingArea passados a em falta: 28
Linhas finais: 13579


### Preço esperado por regressão
Para saber se um imóvel foi vendido acima ou abaixo do normal, estima-se o preço esperado com uma regressão e compara-se com o preço real.

- **Treino só com vendas concluídas** (`S`, `SP`, `SA`). Em `PI` e `VB` o `Price` é uma licitação: fica, em mediana, cerca de 10% abaixo do esperado e é muito mais disperso, por isso não entra no modelo.
- **Alvo:** `log(Price)`. O preço é muito assimétrico, e em log o erro passa a ser relativo (percentagem) em vez de absoluto.
- **Variáveis:** `Rooms`, `Bathroom`, `Car`, `Distance`, `Landsize`, `Type`, `Suburb` e o mês da venda. `BuildingArea` e `YearBuilt` ficam de fora por terem 40-48% de valores em falta. Os poucos `Car` em falta são preenchidos com a mediana.
- **Efeitos não lineares e por tipo de imóvel.** O efeito de `Landsize` e `Distance` não é uma reta, por isso entram como *splines* (curvas por troços), separadas por `Type`: nas unidades o `Landsize` é muitas vezes o terreno do prédio inteiro e não pode valer o mesmo que numa casa. `Rooms` entra combinado com `Type` (ex.: casa com 3 divisões, unidade com 2), e `Bathroom` e `Car` como categorias. O mês da venda capta a evolução do mercado entre 2016 e 2017.
- **Regressão Ridge**, uma regressão linear com uma pequena penalização, por causa das centenas de colunas criadas (`Suburb`, mês, splines).
- **Previsões *out-of-fold*** (validação cruzada com 5 partes): o preço esperado de cada venda vem de um modelo que não viu essa venda. Assim, um preço anormal não "puxa" o modelo para si próprio.

Numa primeira versão, com as variáveis em linha reta e sem o mês, o R² era 0,79 e o erro mediano 15%. A árvore de decisão do `02_classify` aprendia sobretudo os erros dessa regressão (por exemplo, o `Landsize` das unidades), em vez do que faz um imóvel vender acima ou abaixo do valor.

In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer

# Landsize e Distance separados por tipo de imóvel (0 nos outros tipos)
dados["LogLandsize"] = np.log1p(dados["Landsize"])
for tipo in ["h", "t", "u"]:
    dados[f"LogLandsize_{tipo}"] = np.where(dados["Type"] == tipo, dados["LogLandsize"], 0)
    dados[f"Distance_{tipo}"] = np.where(dados["Type"] == tipo, dados["Distance"], 0)

# Divisões por tipo, casas de banho e lugares de garagem como categorias (6 ou mais juntos)
categoria = lambda col: dados[col].fillna(dados[col].median()).clip(upper=6).astype(int).astype(str)
dados["TipoDivisoes"] = dados["Type"] + categoria("Rooms")
dados["BathroomCat"] = categoria("Bathroom")
dados["CarCat"] = categoria("Car")
dados["MesVenda"] = dados["Date"].dt.to_period("M").astype(str)

spline_cols = ["Distance", "LogLandsize_h", "LogLandsize_t", "LogLandsize_u", "Distance_h", "Distance_t", "Distance_u"]
cat_cols = ["TipoDivisoes", "BathroomCat", "CarCat", "Suburb", "MesVenda"]
colunas_auxiliares = ["LogLandsize", "TipoDivisoes", "BathroomCat", "CarCat", "MesVenda"] + spline_cols[1:]

modelo = make_pipeline(
    ColumnTransformer([
        ("spline", SplineTransformer(n_knots=6, degree=3), spline_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]),
    Ridge(alpha=1.0),
)

treino = dados[dados["Vendido"]]
X, y = treino[spline_cols + cat_cols], np.log(treino["Price"])
y_prev = cross_val_predict(modelo, X, y, cv=KFold(5, shuffle=True, random_state=42))

r2 = 1 - ((y - y_prev) ** 2).sum() / ((y - y.mean()) ** 2).sum()
erro_pct = (np.exp(np.abs(y - y_prev)) - 1) * 100
print(f"R² (log do preço, out-of-fold): {r2:.3f}")
print(f"Erro mediano: {np.median(erro_pct):.1f}% do preço")

R² (log do preço, out-of-fold): 0.851
Erro mediano: 12.1% do preço


### Classe de preço: baixo, normal ou alto
O desvio de cada venda é a diferença, em %, entre o preço real e o esperado. O limite é **±12%**, próximo do erro mediano do modelo: uma venda só é considerada alta ou baixa se estiver mais longe do esperado do que o modelo costuma errar.
- `baixo`: vendida mais de 12% abaixo do esperado
- `normal`: entre −12% e +12%
- `alto`: vendida mais de 12% acima do esperado

As classes não ficam do mesmo tamanho (cerca de metade `normal`). Na classificação isso deve ser tido em conta, por exemplo com `class_weight="balanced"` e divisões estratificadas.

As linhas `PI`/`VB` ficam no dataset, mas sem classe (`NaN`): não houve venda, por isso não há preço para classificar.

In [33]:
dados["PrecoEsperado"] = np.nan
dados.loc[treino.index, "PrecoEsperado"] = np.exp(y_prev).round(0)
dados["DesvioPreco"] = (dados["Price"] / dados["PrecoEsperado"] - 1) * 100  # em %

LIMITE = 12  # %
dados["ClassePreco"] = pd.cut(dados["DesvioPreco"], [-np.inf, -LIMITE, LIMITE, np.inf],
                              labels=["baixo", "normal", "alto"])

print(f"baixo: desvio < -{LIMITE}% | normal: entre -{LIMITE}% e +{LIMITE}% | alto: desvio > +{LIMITE}%")
contagem = dados["ClassePreco"].value_counts(dropna=False).to_frame()
contagem["pct_vendas"] = (dados["ClassePreco"].value_counts(normalize=True) * 100).round(1)
contagem

baixo: desvio < -12% | normal: entre -12% e +12% | alto: desvio > +12%


,count,pct_vendas
ClassePreco,,
normal,5653,52.3
NaN,2762,NaN
alto,2675,24.7
baixo,2489,23.0


In [34]:
# Desvio por classe
dados.groupby("ClassePreco", observed=True)["DesvioPreco"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
ClassePreco,,,,,,,,
baixo,2489.0,-21.0,8.0,-75.4,-24.5,-19.0,-15.1,-12.0
normal,5653.0,-0.5,6.6,-12.0,-6.0,-0.6,5.0,12.0
alto,2675.0,28.2,19.2,12.0,16.3,22.5,33.2,344.8


In [35]:
# As classes estão equilibradas entre tipos de imóvel e regiões? (percentagem por linha)
display((pd.crosstab(dados["Type"], dados["ClassePreco"], normalize="index") * 100).round(1))
(pd.crosstab(dados["Regionname"], dados["ClassePreco"], normalize="index") * 100).round(1)

ClassePreco,baixo,normal,alto
Type,,,
h,22.0,53.9,24.1
t,22.3,54.6,23.0
u,26.6,45.8,27.6


ClassePreco,baixo,normal,alto
Regionname,,,
Eastern Metropolitan,20.8,54.7,24.5
Eastern Victoria,19.6,58.7,21.7
Northern Metropolitan,22.0,54.9,23.1
Northern Victoria,23.1,69.2,7.7
South-Eastern Metropolitan,17.9,58.0,24.1
Southern Metropolitan,24.6,47.5,27.9
Western Metropolitan,23.8,53.1,23.1
Western Victoria,36.7,43.3,20.0


In [36]:
# Exemplos: as vendas mais acima e mais abaixo do esperado
cols = ["Suburb", "Address", "Type", "Rooms", "Landsize", "Price", "PrecoEsperado", "DesvioPreco", "ClassePreco"]
display(dados.nlargest(5, "DesvioPreco")[cols].round(1))
dados.nsmallest(5, "DesvioPreco")[cols].round(1)

,Suburb,Address,Type,Rooms,Landsize,Price,PrecoEsperado,DesvioPreco,ClassePreco
233,Armadale,11/17 Myamyn St,u,2,0.0,3625000.0,814986.0,344.8,alto
5631,South Yarra,18 Avoca St,h,4,292.0,5700000.0,2081266.0,173.9,alto
3580,Kew,51 Rowland St,h,2,697.0,4800000.0,1825160.0,163.0,alto
2117,Collingwood,12/120 Cambridge St,u,2,0.0,1645000.0,625626.0,162.9,alto
8237,Port Melbourne,9/23 Beaconsfield Pde,u,2,0.0,2250000.0,864136.0,160.4,alto


,Suburb,Address,Type,Rooms,Landsize,Price,PrecoEsperado,DesvioPreco,ClassePreco
4579,Parkville,9/18 Lennon St,h,1,0.0,307000.0,1246303.0,-75.4,baixo
5556,South Melbourne,58C Napier St,h,3,2325.0,770000.0,3108573.0,-75.2,baixo
13249,North Melbourne,5 Hardwicke St,h,2,1044.0,516000.0,2063289.0,-75.0,baixo
13481,Malvern,7 Northbrook Av,h,4,633.0,860000.0,3054356.0,-71.8,baixo
3944,Maribyrnong,23/4 Wests Rd,h,3,4977.0,456000.0,1514223.0,-69.9,baixo


### Guardar o dataset preparado
Para ser usado no `02_classify` e no `03_cluster`. A pasta `data/` está no `.gitignore`, por isso o ficheiro é gerado ao correr este notebook.

In [37]:
saida = "data/melbourne/melb_clean.csv"
dados.drop(columns=colunas_auxiliares).to_csv(saida, index=False)
print("Guardado em", saida, dados.shape)

Guardado em data/melbourne/melb_clean.csv (13579, 36)
